# GPU Environment Check

Use this notebook in Google Colab after selecting **Runtime → Change runtime type → T4 GPU**, or in another CUDA-enabled environment. This Windows machine currently exposes Intel UHD graphics and TensorFlow reports no GPU; those devices cannot run this CUDA test.

## 1. Check GPU Availability

Check whether TensorFlow or PyTorch can access a CUDA GPU. A CPU-only runtime is reported explicitly.

In [ ]:
import importlib.util
import platform

print("Platform:", platform.platform())

tf_gpus = []
if importlib.util.find_spec("tensorflow"):
    import tensorflow as tf
    tf_gpus = tf.config.list_physical_devices("GPU")
    print("TensorFlow:", tf.__version__)
    print("TensorFlow GPUs:", tf_gpus)
else:
    print("TensorFlow is not installed")

torch_cuda_available = False
if importlib.util.find_spec("torch"):
    import torch
    torch_cuda_available = torch.cuda.is_available()
    print("PyTorch:", torch.__version__)
    print("PyTorch CUDA available:", torch_cuda_available)
else:
    print("PyTorch is not installed")

gpu_available = bool(tf_gpus) or torch_cuda_available
print("\nGPU available:", gpu_available)

## 2. Display GPU Details

Show the accelerator name and memory where available, plus the NVIDIA driver/CUDA report from `nvidia-smi`.

In [ ]:
import subprocess

if torch_cuda_available:
    properties = torch.cuda.get_device_properties(0)
    print("GPU:", properties.name)
    print("Total GPU memory (GiB):", round(properties.total_memory / (1024 ** 3), 2))
elif tf_gpus:
    for gpu in tf_gpus:
        details = tf.config.experimental.get_device_details(gpu)
        print("GPU:", details.get("device_name", gpu.name))
        print("Details:", details)
else:
    print("No CUDA GPU is visible to this runtime.")

try:
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
        capture_output=True,
        text=True,
        check=True,
    )
    print("\nnvidia-smi (name, memory, driver):")
    print(result.stdout.strip())
except (FileNotFoundError, subprocess.CalledProcessError):
    print("nvidia-smi is unavailable; this runtime may not have an NVIDIA GPU/driver.")

## 3. Test GPU with a Simple Computation

Run matrix multiplication only on a detected GPU. If no GPU is available, the cell reports that the test was skipped instead of silently using the CPU.

In [ ]:
import time

if not gpu_available:
    print("Skipped: no CUDA GPU is available in this environment.")
elif tf_gpus:
    with tf.device(tf_gpus[0].name):
        left = tf.random.uniform((1024, 1024))
        right = tf.random.uniform((1024, 1024))
        started = time.perf_counter()
        product = tf.linalg.matmul(left, right)
        _ = product.numpy()
        elapsed = time.perf_counter() - started
    print("TensorFlow matrix multiplication:", product.shape)
    print("Tensor location:", product.device)
    print("Elapsed seconds:", round(elapsed, 4))
elif torch_cuda_available:
    device = torch.device("cuda:0")
    left = torch.rand((1024, 1024), device=device)
    right = torch.rand((1024, 1024), device=device)
    torch.cuda.synchronize()
    started = time.perf_counter()
    product = left @ right
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - started
    print("PyTorch matrix multiplication:", tuple(product.shape))
    print("Tensor device:", product.device)
    print("Elapsed seconds:", round(elapsed, 4))

## 4. List Installed GPU Libraries

Report installed TensorFlow/PyTorch versions and their CUDA/cuDNN build information. CUDA toolkit and driver details are shown only when the runtime provides them.

In [ ]:
from importlib import metadata

for package in ("tensorflow", "torch", "nvidia-cuda-runtime-cu12", "nvidia-cudnn-cu12"):
    try:
        print(f"{package}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package}: not installed")

if importlib.util.find_spec("tensorflow"):
    build = tf.sysconfig.get_build_info()
    print("TensorFlow CUDA build:", build.get("is_cuda_build", "unknown"))
    print("TensorFlow CUDA version:", build.get("cuda_version", "not available"))
    print("TensorFlow cuDNN version:", build.get("cudnn_version", "not available"))

if importlib.util.find_spec("torch"):
    print("PyTorch CUDA build:", torch.version.cuda or "CPU-only")
    print("PyTorch cuDNN version:", torch.backends.cudnn.version() or "not available")

print("\nReady for GPU training:", gpu_available)

## Use for PolyScope

For training, run `train_model.py` only after this notebook confirms a visible GPU and the dataset folders are present. A GPU check only confirms hardware execution; it does not validate model quality. This polyp-segmentation model is research-only and must not guide patient care.

## Train PolyScope in Colab

To train, first upload the project’s training files and datasets to Google Drive under `MyDrive/PolyScope` with this layout:

```text
PolyScope/
  train_model.py
  Kvasir-seg/images/
  Kvasir-seg/masks/
  data/polyp/train/
  data/polyp/val/
  data/no_polyp/
```

Do not upload `users.json`, `instance/`, `.env`, or patient data. In Colab, choose a GPU runtime before running the cells below. The dataset is copied to Colab’s temporary disk for faster training.

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

if not tf_gpus:
    raise RuntimeError("TensorFlow cannot see a GPU. In Colab, select Runtime > Change runtime type > T4 GPU, then rerun the GPU check.")

drive.mount("/content/drive")
project_on_drive = Path("/content/drive/MyDrive/PolyScope")
required_dirs = [
    project_on_drive / "Kvasir-seg" / "images",
    project_on_drive / "Kvasir-seg" / "masks",
    project_on_drive / "data" / "polyp" / "train",
    project_on_drive / "data" / "polyp" / "val",
    project_on_drive / "data" / "no_polyp",
]
missing_dirs = [str(path) for path in required_dirs if not path.is_dir()]
if not (project_on_drive / "train_model.py").is_file():
    missing_dirs.append(str(project_on_drive / "train_model.py"))
if missing_dirs:
    raise FileNotFoundError("Missing project files/folders:\n" + "\n".join(missing_dirs))

training_dir = Path("/content/polyscope_training")
if training_dir.exists():
    shutil.rmtree(training_dir)
training_dir.mkdir(parents=True)
shutil.copy2(project_on_drive / "train_model.py", training_dir / "train_model.py")
for relative_dir in (
    "Kvasir-seg/images",
    "Kvasir-seg/masks",
    "data/polyp/train",
    "data/polyp/val",
    "data/no_polyp",
):
    source = project_on_drive / relative_dir
    destination = training_dir / relative_dir
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(source, destination)

print("Training files copied to:", training_dir)
for relative_dir in ("data/polyp/train", "data/polyp/val", "data/no_polyp"):
    count = sum(path.is_file() for path in (training_dir / relative_dir).iterdir())
    print(f"{relative_dir}: {count} files")

### Train

The trainer verifies the renamed Kvasir image/mask pairs, uses empty masks for the normal images, and reports validation loss and polyp IoU. Set `TRAIN_EPOCHS` below before starting; training may take a while even on a free GPU. Do not treat these validation metrics as clinical validation.

In [ ]:
import os
import subprocess
import sys

TRAIN_EPOCHS = 20
training_env = os.environ.copy()
training_env["TRAIN_EPOCHS"] = str(TRAIN_EPOCHS)
training_env["TF_CPP_MIN_LOG_LEVEL"] = "2"

subprocess.run(
    [sys.executable, "-u", "train_model.py"],
    cwd=training_dir,
    env=training_env,
    check=True,
)

trained_model = training_dir / "models" / "polyp_segmenter.keras"
if not trained_model.is_file():
    raise FileNotFoundError("Training completed without creating the expected model file")
print("Training completed:", trained_model)
print("Model size (MiB):", round(trained_model.stat().st_size / (1024 ** 2), 2))

### Save the Weights

The trained model is copied back to Drive. The optional download cell saves a local copy; put that file at the app's configured `MODEL_PATH` before expecting the dashboard to enable analysis.

In [ ]:
from google.colab import files

model_in_drive = project_on_drive / "models" / "polyp_segmenter.keras"
model_in_drive.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(trained_model, model_in_drive)
print("Saved to Google Drive:", model_in_drive)

files.download(str(model_in_drive))